# Task 5: Exploratory Data Analysis

This notebook covers possible data errors and outliers, relationships between variables, and descriptive statistics. Visualizations will be

In [1]:
# Load the cleaned transaction data and preview its shape, columns, and first rows.
import pandas as pd

df = pd.read_parquet("../data/online_retail_ii_cleaned_v3.parquet")

# Create a completed-purchases view for transaction analysis; keep the full data unchanged.
purchases = df.loc[
    ~df["Is_Cancellation"] & df["Quantity"].gt(0)
].copy()
analysis_cols = ["Quantity", "Price", "Total_Price"]

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (794183, 12)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country', 'Is_Cancellation', 'Net_Quantity', 'Total_Price', 'CustomerMonth']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Is_Cancellation,Net_Quantity,Total_Price,CustomerMonth
0,499763,20682,RED SPOTTY CHILDS UMBRELLA,1,2010-03-02 13:08:00,3.25,12346,United Kingdom,False,1,3.25,2010-03
1,499763,20679,EDWARDIAN PARASOL RED,1,2010-03-02 13:08:00,5.95,12346,United Kingdom,False,2,5.95,2010-03
2,499763,15056N,EDWARDIAN PARASOL NATURAL,1,2010-03-02 13:08:00,5.95,12346,United Kingdom,False,3,5.95,2010-03
3,499763,15056BL,EDWARDIAN PARASOL BLACK,1,2010-03-02 13:08:00,5.95,12346,United Kingdom,False,4,5.95,2010-03
4,499763,15056P,EDWARDIAN PARASOL PINK,1,2010-03-02 13:08:00,5.95,12346,United Kingdom,False,5,5.95,2010-03


In [2]:
# Check dataset size, exact duplicate rows, missing values, data types, and unique counts.
print("Rows and columns:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

quality_check = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique_values": df.nunique()
})
quality_check

Rows and columns: (794183, 12)
Exact duplicate rows: 0


,dtype,missing,unique_values
Invoice,string,0,43877
StockCode,str,0,4635
Description,string,0,5286
Quantity,int64,0,226
InvoiceDate,datetime64[ns],0,40621
Price,float64,0,398
Customer ID,int64,0,5875
Country,string,0,41
Is_Cancellation,bool,0,2
Net_Quantity,int64,0,52221


**Interpretation:** This cleaned dataset contains 794,183 line-item rows. The quality check found no missing values or exact duplicate rows. This confirms those basic checks pass; it does not guarantee that every value is valid.

In [3]:
# Summarize transaction measures for completed purchases, excluding IDs and cumulative totals.
stats = purchases[analysis_cols].agg(["mean", "median", "std", "var"]).T
stats["mode"] = purchases[analysis_cols].mode().iloc[0]
stats

,mean,median,std,var,mode
Quantity,11.199651,6.00,19.814960,392.632632,1.00
Price,2.833140,1.95,2.650900,7.027270,1.25
Total_Price,19.638986,12.45,39.340312,1547.660154,15.00


**Interpretation:** For all three completed-purchase measures, the mean is above the median, suggesting right-skew: a smaller number of larger values pull the average upward. Quantity has a mean of about 11.2 but a median of 6; typical line quantity is closer to the median. The standard deviation describes spread in the original units, while variance is the squared spread.

In [4]:
# Flag IQR outliers in completed-purchase measures; investigate them rather than dropping automatically.
outlier_results = []

for col in analysis_cols:
    q1 = purchases[col].quantile(0.25)
    q3 = purchases[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    count = ((purchases[col] < lower_bound) | (purchases[col] > upper_bound)).sum()

    outlier_results.append({
        "feature": col,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "possible_outliers": count,
        "percent": 100 * count / len(purchases)
    })

pd.DataFrame(outlier_results).set_index("feature")

,lower_bound,upper_bound,possible_outliers,percent
feature,,,,
Quantity,-13.000,27.000,51077,6.577036
Price,-2.500,7.500,63131,8.129195
Total_Price,-16.875,41.325,62461,8.042921


**Interpretation:** The IQR rule flags about 6.6% of purchase lines for quantity, 8.1% for price, and 8.0% for line value. These are candidates for review, not confirmed errors. The bounds are statistical cutoffs; an unusual order size or price can still be a legitimate transaction.

In [5]:
# Compare cancellation invoice prefixes with negative quantities; these signals can overlap.
c_invoice = df["Invoice"].astype("string").str.startswith("C", na=False)
negative_qty = df["Quantity"] < 0

pd.crosstab(c_invoice, negative_qty, margins=True)

Quantity,False,True,All
Invoice,,,
False,776596,0,776596
True,0,17587,17587
All,776596,17587,794183


In [6]:
# Count cancellation line items and unique invoices; these counts are not interchangeable.
cancellations = df.loc[df["Is_Cancellation"]].copy()

cancellation_summary = pd.Series({
    "Cancellation line rows": len(cancellations),
    "Distinct cancellation invoices": cancellations["Invoice"].nunique(),
    "Percent of all line rows": 100 * len(cancellations) / len(df),
    "Sum of cancellation line values": cancellations["Total_Price"].sum()
})
cancellation_summary.rename("value").to_frame()

,value
Cancellation line rows,17587.000000
Distinct cancellation invoices,7283.000000
Percent of all line rows,2.214477
Sum of cancellation line values,-353485.810000


**Interpretation:** The 17,587 cancellation rows belong to 7,283 distinct invoices, so multiple line items can belong to one canceled invoice. Their summed line value is negative because the cancellation entries reverse sales value. They make up about 2.2% of all line rows and are kept in `df` but excluded from completed-purchase statistics.

In [7]:
# Quantify the relationship between unit price and quantity using linear and rank correlations.
quantity_price_correlations = pd.DataFrame({
    "Pearson": purchases[["Quantity", "Price"]].corr(method="pearson").loc["Quantity", "Price"],
    "Spearman": purchases[["Quantity", "Price"]].corr(method="spearman").loc["Quantity", "Price"]
}, index=["Quantity vs. Price"])
quantity_price_correlations

,Pearson,Spearman
Quantity vs. Price,-0.230188,-0.42624


**Interpretation:** Both correlation values are negative, indicating that higher quantities tend to be associated with lower unit prices. The Spearman relationship (-0.43) is stronger than the Pearson relationship (-0.23), suggesting the overall ranked trend is clearer than a straight-line pattern. This is association, not evidence that quantity causes price to change.